# Episode 16: Before Transformers: Counting Words
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2016%20-%20Counting%20Words.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers and the running-sentence experiment from Episode 16.

* **Part 1** is the Math Lab: counting is training (the counts minimize the loss), a hand-set toy with add-one smoothing, the size of count tables, and add-k and Kneser-Ney. It uses `fractions` and `math` only and runs in a second.
* **Part 2** is **our own experiment**: WikiText-103 (raw), our light detokenizer, GPT-2 tokens (tiktoken `gpt2`), one released train, validation and test split, no random number anywhere. It downloads about 300 MB, takes a few minutes and needs about 8 GB of RAM.
* **Part 3** runs GPT-2 small (548 MB of weights, downloaded once) on the running sentence and on Episode 15's 2,048 sentences. The step that scores the whole test stream takes about 9 minutes on a CPU; switch it off with `RUN_GPT2_TEST = False`.

The results are **ours: one corpus, one tokenizer, one released split. They are not facts about language, and the GPT-2 comparison is not a ranking** (GPT-2 small was not trained on this training split; data, context length and size all differ). Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the episode or in the code is wrong.

## Part 1: the Math Lab
### Step 1 and Step 2: counting is training, on a toy
Three documents, words as tokens, no start token (Episode 10): "the cat sat on the mat", "the dog sat on the rug", "the cat ate the fish". The bigram table is the counts divided by the previous token's count. We check the numbers in the episode with exact fractions, then check that **no other table has a lower training loss**: the loss splits by previous token as N × L = Σ count(previous) × H(q, p), where q is that token's observed next-token mix, and H(q, p) = H(q) + D_KL(q‖p) is smallest at p = q.

In [1]:
import math, itertools, json, time, os, sys, hashlib, re
from collections import Counter
from fractions import Fraction as F

checks = 0
def ok(cond, msg=""):
    global checks
    assert cond, msg
    checks += 1

docs = ["the cat sat on the mat", "the dog sat on the rug", "the cat ate the fish"]
words = [d.split() for d in docs]
pairs = [(a, b) for w in words for a, b in zip(w[:-1], w[1:])]
Vtoy = sorted({x for w in words for x in w})
N = len(pairs)
ok(N == 14 and len(Vtoy) == 9)                                  # N = 14 pairs, |V| = 9
c2 = Counter(pairs)
ctx = Counter(a for a, b in pairs)
ok(len(c2) == 11 and ctx["the"] == 6 and ctx["cat"] == 2)
p_mle = {ab: F(c, ctx[ab[0]]) for ab, c in c2.items()}
ok(p_mle[("the", "cat")] == F(2, 6) and all(p_mle[("the", w)] == F(1, 6) for w in ("mat", "dog", "rug", "fish")))
ok(p_mle[("cat", "sat")] == F(1, 2) and p_mle[("cat", "ate")] == F(1, 2))
others = [a for a in ctx if a not in ("the", "cat")]
ok(all(sum(1 for (x, b) in c2 if x == a) == 1 for a in others))   # every other context has one follower

def loss(table):
    """mean of -ln p(x_t | x_{t-1}) over the N training pairs"""
    return -sum(math.log(float(table[ab])) for ab in pairs) / N

L_mle = loss(p_mle)
prod = F(1)
for ab in pairs:
    prod *= p_mle[ab]
ok(prod == F(1, 6 ** 6))                                         # the training product is 6^-6
ok(abs(L_mle - 0.768) < 5e-4 and round(L_mle, 3) == 0.768)       # L = 0.768 nats
ok(abs(math.exp(L_mle) - 6 ** (6 / 14)) < 1e-12 and round(math.exp(L_mle), 2) == 2.16)   # perplexity 2.16 = 6^(6/14)
print("toy: N =", N, " |V| =", len(Vtoy), " L =", round(L_mle, 5), "nats  perplexity =", round(math.exp(L_mle), 4))

# the identity N x L = sum over previous tokens of count(previous) x H(q, p), for ANY table p that covers the pairs
def H_cross(q, p):
    return -sum(float(qq) * math.log(float(p[w])) for w, qq in q.items() if qq > 0)
def H_ent(q):
    return -sum(float(qq) * math.log(float(qq)) for qq in q.values() if qq > 0)
def kl(q, p):
    return sum(float(qq) * math.log(float(qq) / float(p[w])) for w, qq in q.items() if qq > 0)

rows_q = {a: {b: F(c2[(a, b)], ctx[a]) for (x, b) in c2 if x == a} for a in ctx}
def full_table(rowdict):
    return {(a, b): rowdict[a][b] for a in rowdict for b in rowdict[a]}

# any table: start from the MLE and move some mass from a seen follower to another token (seen or not)
def perturbed(a, b_from, b_to, eps):
    rd = {x: {w: rows_q[x].get(w, F(0)) for w in Vtoy} for x in rows_q}
    shift = eps * rd[a][b_from]
    rd[a][b_from] -= shift
    rd[a][b_to] += shift
    return rd
n_pert = 0
for a in ctx:
    for b_from in rows_q[a]:
        for b_to in Vtoy:
            if b_to == b_from:
                continue
            for eps in (F(1, 50), F(1, 10)):
                rd = perturbed(a, b_from, b_to, eps)
                tab = {(x, w): rd[x][w] for x in rd for w in Vtoy}
                if any(tab[ab] == 0 for ab in pairs):
                    continue
                n_pert += 1
                ok(loss(tab) > L_mle, (a, b_from, b_to, eps))              # every perturbation raises the training loss
                lhs = N * loss(tab)
                rhs = sum(ctx[x] * (H_ent(rows_q[x]) + kl(rows_q[x], {w: rd[x][w] for w in Vtoy})) for x in ctx)
                ok(abs(lhs - rhs) < 1e-9, "N x L identity")
ok(n_pert >= 110)
lhs = N * L_mle
rhs = sum(ctx[x] * H_ent(rows_q[x]) for x in ctx)
ok(abs(lhs - rhs) < 1e-12)                                         # at p = q the identity is N x L = sum count x H(q)
print(n_pert, "perturbed tables, every one has a higher training loss; N x L = sum of count x H(q, p) holds on each")

toy: N = 14  |V| = 9  L = 0.7679 nats  perplexity = 2.1552
176 perturbed tables, every one has a higher training loss; N x L = sum of count x H(q, p) holds on each


### Step 2, continued: a held-out sentence, unsmoothed and with add-one
Held-out: "the dog ate the fish". The pair (dog, ate) never occurred, and "dog" occurred once, so the maximum likelihood estimate is 0/1. One zero makes the product 0 and the perplexity infinite. Add-one smoothing: (count + 1) / (count(previous) + |V|).

In [2]:
held = [("the", "dog"), ("dog", "ate"), ("ate", "the"), ("the", "fish")]
p_held = [p_mle.get(ab, F(0)) for ab in held]
ok(p_held == [F(1, 6), F(0), F(1), F(1, 6)])
ok(math.prod(p_held) == 0)                                         # product 0, so perplexity is infinite
Vn = len(Vtoy)
add1 = [F(c2.get(ab, 0) + 1, ctx.get(ab[0], 0) + Vn) for ab in held]
ok(add1 == [F(2, 15), F(1, 10), F(1, 5), F(2, 15)])
ok(math.prod(add1) == F(2, 5625))
pp1 = float(math.prod(add1)) ** -0.25                                # perplexity = (product)^(-1/4)
ok(round(pp1, 2) == 7.28 and abs(pp1 - (5625 / 2) ** 0.25) < 1e-12)
ok(add1[1] * 2 == F(c2[("dog", "sat")] + 1, ctx["dog"] + Vn))      # the unseen pair gets half of what the seen (dog, sat) gets
ok(sum(F(1, 10) for w in Vtoy if c2.get(("dog", w), 0) == 0) == F(8, 10))   # 8/10 of the mass after "dog" sits on unseen tokens
print("held-out: unsmoothed product 0 (perplexity infinite); add-one", [str(x) for x in add1], "product", math.prod(add1), "perplexity", round(pp1, 4))

held-out: unsmoothed product 0 (perplexity infinite); add-one ['2/15', '1/10', '1/5', '2/15'] product 2/5625 perplexity 7.2824


### Step 3: size (exact)
Each extra token of context multiplies a dense table by |V| = 50,257, GPT-2's vocabulary. Python integers are exact.

In [3]:
V = 50257
ok(V ** 2 == 2_525_766_049 and V ** 3 == 126_937_424_324_593 and V ** 4 == 6_379_494_134_281_070_401)
ok(V ** 4 < 2 ** 63)                                                # int64 keys are safe for 4-grams
ok(4 * V ** 2 == 10_103_064_196 and 4 * V ** 3 == 507_749_697_298_372)   # bytes at 4 bytes a count: 10.1 GB and about 508 TB
ok(round(4 * V ** 2 / 1e9, 1) == 10.1 and round(4 * V ** 3 / 1e12) == 508)
ok(10_000 ** 3 == 10 ** 12 and F(3, 12) == F(1, 4) and F(3 + 1, 12 + 10_000) == F(4, 10_012) and round(4 / 10_012, 4) == 0.0004)   # the check question
ok(round(100 * (V - 20) / (100 + V), 1) == 99.8)                    # add-one on a context seen 100 times with 20 followers: 99.8% on unseen tokens
print("|V|^2 =", f"{V**2:,}", " |V|^3 =", f"{V**3:,}", " |V|^4 =", f"{V**4:,}")
print("dense bigram table:", f"{4*V**2:,}", "bytes   trigram table:", f"{4*V**3:,}", "bytes")

|V|^2 = 2,525,766,049  |V|^3 = 126,937,424,324,593  |V|^4 = 6,379,494,134,281,070,401
dense bigram table: 10,103,064,196 bytes   trigram table: 507,749,697,298,372 bytes


### Step 4: add-k and Kneser-Ney sum to 1
Add-k: (count(x_{t−1}, x_t) + k) / (count(x_{t−1}) + k|V|) sums to 1 over V. Interpolated Kneser-Ney: p_KN(w | a) = max(count(a, w) − d, 0) / count(a) + d × (different followers of a) / count(a) × p_cont(w), with p_cont(w) = (different tokens that precede w) / (different bigrams). For a context that occurs, the first part sums to 1 minus the second part's weight and p_cont sums to 1, so p_KN sums to 1. Exact fractions on the toy, discount d = 3/4.

In [4]:
for k in (F(1), F(1, 2), F(1, 1000)):
    for a in Vtoy:                                                   # every token as a context, including ones that never occur
        s = sum(F(c2.get((a, w), 0)) + k for w in Vtoy) / (ctx.get(a, 0) + k * Vn)
        ok(s == 1, (k, a))
d = F(3, 4)
n_bi = len(c2)                                                       # different bigrams
pred = {w: len({a for (a, b) in c2 if b == w}) for w in Vtoy}        # different tokens preceding w
p_cont = {w: F(pred[w], n_bi) for w in Vtoy}
ok(sum(p_cont.values()) == 1 and n_bi == 11)
for a in ctx:
    foll = len({b for (x, b) in c2 if x == a})
    p_kn = {w: max(F(c2.get((a, w), 0)) - d, F(0)) / ctx[a] + d * foll / ctx[a] * p_cont[w] for w in Vtoy}
    ok(sum(p_kn.values()) == 1, a)                                   # every context that occurs: exactly 1
    ok(all(c >= d for (x, w), c in c2.items() if x == a))            # the condition: every count is at least the discount
print("add-k sums to 1 over V (k = 1, 1/2, 1/1000) and interpolated Kneser-Ney sums to 1 at all", len(ctx), "contexts that occur (exact fractions)")
MATH_CHECKS = checks
print("MATH LAB CHECKS PASSED:", MATH_CHECKS, "checks")

add-k sums to 1 over V (k = 1, 1/2, 1/1000) and interpolated Kneser-Ney sums to 1 at all 6 contexts that occur (exact fractions)
MATH LAB CHECKS PASSED: 415 checks


## Part 2: our experiment, WikiText-103 and GPT-2 tokens
**Ours: one corpus, one tokenizer, one released split, no random number anywhere.** Data: Hugging Face `Salesforce/wikitext`, config `wikitext-103-raw-v1` (raw files, before the `<unk>` tokens were added; `wikitext-103-v1` is the Moses-tokenized release with 103,227,021 training words). Each non-empty row (a heading or a paragraph) is detokenized with our own light rules (it joins ` @-@ `, ` @,@ `, ` @.@ `, spaces before punctuation and clitics, parentheses), encoded on its own with tiktoken `gpt2`, and the rows are joined into one stream per split with no separator. Smaller training sizes are the **first N tokens** of the training stream. k, the interpolation weights and the cache weight are tuned on validation text; Kneser-Ney's discount 0.75 is fixed in advance.

In [5]:
import subprocess, sys
for pkg in ["tiktoken", "huggingface_hub", "pyarrow", "pandas"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import numpy as np, pandas as pd, tiktoken
from huggingface_hub import hf_hub_download

RUN_GPT2_TEST = True        # scoring the whole test stream with GPT-2 small takes about 9 minutes on a CPU
enc = tiktoken.get_encoding("gpt2")
V = 50257
ok(enc.n_vocab == V)
sent_ids = [464, 3797, 3332, 319, 262, 2603, 780, 340, 373, 10032, 13]
ok(enc.encode("The cat sat on the mat because it was tired.") == sent_ids)          # 11 tokens, n = 11, N = 10
ok([enc.decode([i]) for i in sent_ids] == ["The", " cat", " sat", " on", " the", " mat", " because", " it", " was", " tired", "."])
print("tiktoken", tiktoken.__version__, "gpt2 encoding, |V| =", enc.n_vocab)

tiktoken 0.14.0 gpt2 encoding, |V| = 50257


In [6]:
FILES = {"train": ["train-00000-of-00002.parquet", "train-00001-of-00002.parquet"],
         "validation": ["validation-00000-of-00001.parquet"], "test": ["test-00000-of-00001.parquet"]}
SHA_PREFIX = {"train-00000-of-00002.parquet": "74da360f", "train-00001-of-00002.parquet": "ba090ac3",
              "validation-00000-of-00001.parquet": "204929b7", "test-00000-of-00001.parquet": "5f1bea06"}
paths = {}
for split, fs in FILES.items():
    paths[split] = []
    for f in fs:
        p = hf_hub_download("Salesforce/wikitext", "wikitext-103-raw-v1/" + f, repo_type="dataset", local_dir="wikitext_data")
        h = hashlib.sha256(open(p, "rb").read()).hexdigest()
        ok(h.startswith(SHA_PREFIX[f]), ("the file on the hub changed: " + f, h))
        paths[split].append(p)
print("downloaded and hash-checked", sum(len(v) for v in paths.values()), "parquet files")

downloaded and hash-checked 4 parquet files


In [7]:
REPL = [(" @-@ ", "-"), (" @,@ ", ","), (" @.@ ", "."),
        (" ,", ","), (" .", "."), (" ;", ";"), (" :", ":"), (" ?", "?"), (" !", "!"),
        (" 's", "'s"), (" n't", "n't"), (" 're", "'re"), (" 've", "'ve"), (" 'll", "'ll"), (" 'd", "'d"), (" 'm", "'m"),
        ("( ", "("), (" )", ")"), (" \n", "\n")]
def detok(s):
    for a, b in REPL:
        s = s.replace(a, b)
    return s

def tokenize_split(split):
    chunks, starts, off, nrows, nonempty = [], [], 0, 0, 0
    for p in paths[split]:
        texts = pd.read_parquet(p)["text"].tolist()
        nrows += len(texts)
        texts = [detok(t) for t in texts if t.strip() != ""]
        nonempty += len(texts)
        B = 20000
        for i in range(0, len(texts), B):
            batch = enc.encode_ordinary_batch(texts[i:i + B], num_threads=2)
            for ids in batch:
                starts.append(off)
                off += len(ids)
            chunks.append(np.fromiter((x for ids in batch for x in ids), dtype=np.uint16, count=sum(len(ids) for ids in batch)))
    toks = np.concatenate(chunks)
    assert len(toks) == off
    return toks, np.asarray(starts, dtype=np.int64), nrows, nonempty

t0 = time.time()
tr, ls_tr, rows_tr, ne_tr = tokenize_split("train")
va_u16, ls_va, rows_va, ne_va = tokenize_split("validation")
te_u16, ls_te, rows_te, ne_te = tokenize_split("test")
va, te = va_u16.astype(np.int64), te_u16.astype(np.int64)
ok((rows_tr, rows_va, rows_te) == (1_801_350, 3_760, 4_358))                 # rows as released
ok((ne_tr, ne_va, ne_te) == (1_165_029, 2_461, 2_891))                      # non-empty rows
ok(len(tr) == 113_296_181 and len(va) == 237_710 and len(te) == 272_352)    # tokens after our pipeline
ok(int(tr.max()) < V and (len(te) - 1) == 272_351)                           # N = 272,351 scored test targets
print("train", f"{len(tr):,}", "validation", f"{len(va):,}", "test", f"{len(te):,}", "tokens;", round(time.time() - t0), "s")

train 113,296,181 validation 237,710 test 272,352 tokens; 88 s


### The counting library
Keys are integers: a bigram (v, w) is v·|V| + w, a trigram (u, v, w) is (u·|V| + v)·|V| + w, a 4-gram (a, b, c, d) is ((a·|V| + b)·|V| + c)·|V| + d, all below 2⁶³ (|V|⁴ < 2⁶³). Tables are sorted arrays searched with `searchsorted`. Convention (Episode 10): x₁ is context only, N = n − 1 targets; a trigram has no two-token context at t = 2 and uses the bigram estimate there, so every order scores the same targets.

In [8]:
def runs(a):
    n = len(a)
    flag = np.empty(n, dtype=bool); flag[0] = True
    np.not_equal(a[1:], a[:-1], out=flag[1:])
    idx = np.flatnonzero(flag)
    return a[idx], np.diff(idx, append=n).astype(np.int32)

def uni_counts(tok, m):
    out = np.zeros(V, dtype=np.int64)
    B = 10_000_000
    for i in range(1, m, B):
        out += np.bincount(tok[i:min(i + B, m)].astype(np.int64), minlength=V)
    return out

def raw_tables(tok, m):
    t = tok[:m].astype(np.int64)
    k2 = t[:-1] * V; k2 += t[1:]; k2.sort()
    keys2, cnt2 = runs(k2); del k2
    k3 = t[:-2] * V; k3 += t[1:-1]; k3 *= V; k3 += t[2:]
    del t
    k3.sort()
    keys3, cnt3 = runs(k3); del k3
    return keys2, cnt2, keys3, cnt3

def build_tables(tok, m, full=True):
    uni = uni_counts(tok, m)
    keys2, cnt2, keys3, cnt3 = raw_tables(tok, m)
    T = {"m": m, "uni": uni, "Nu": int(m - 1), "keys2": keys2, "cnt2": cnt2, "keys3": keys3}
    v2 = keys2 // V
    T["c2tot"] = np.bincount(v2, weights=cnt2, minlength=V).astype(np.int64)     # count(v) as a context
    T["n2w"] = np.bincount(v2, minlength=V).astype(np.int64)                      # different followers of v
    del v2
    T["stats"] = {"m": int(m), "N_pairs": int(m - 1), "distinct_uni": int(np.count_nonzero(uni)), "distinct_bi": int(len(keys2)),
                  "distinct_tri": int(len(keys3)), "bi_singletons": int((cnt2 == 1).sum()), "tri_singletons": int((cnt3 == 1).sum())}
    T["cs3"] = np.concatenate([[0], np.cumsum(cnt3, dtype=np.int64)])
    del cnt3
    if full:
        T["a1"] = np.bincount(keys2 % V, minlength=V).astype(np.int64)            # different tokens preceding w
        T["A1"] = int(len(keys2)); T["nz1"] = int(np.count_nonzero(T["a1"]))
        k23 = np.sort(keys3 % (V * V))
        keys_a2, cnt_a2 = runs(k23); del k23                                      # a2(v, w) = different u with trigram (u, v, w)
        T["keys_a2"], T["cnt_a2"] = keys_a2, cnt_a2
        v_a2 = keys_a2 // V
        T["tot_a2"] = np.bincount(v_a2, weights=cnt_a2, minlength=V).astype(np.int64)
        T["nw_a2"] = np.bincount(v_a2, minlength=V).astype(np.int64)
        del v_a2
        T["nzu"] = int(np.count_nonzero(uni))
    return T

def look(keys, vals, q):
    i = np.searchsorted(keys, q); j = np.minimum(i, len(keys) - 1)
    return np.where((i < len(keys)) & (keys[j] == q), vals[j], 0)

def look_cs(keys, cs, q):
    i = np.searchsorted(keys, q); j = np.minimum(i, len(keys) - 1)
    return np.where((i < len(keys)) & (keys[j] == q), cs[j + 1] - cs[j], 0)

def components(T, u, v, w, has_u):
    u = np.where(has_u, u, 0)
    C = {"w": w, "v": v, "has_u": has_u}
    C["uni_w"] = T["uni"][w]
    C["c2"] = look(T["keys2"], T["cnt2"], v * V + w)
    C["ctx2"] = T["c2tot"][v]
    C["n2w_v"] = T["n2w"][v]
    key_ctx = (u * V + v) * V
    C["c3"] = np.where(has_u, look_cs(T["keys3"], T["cs3"], key_ctx + w), 0)
    lo = np.searchsorted(T["keys3"], key_ctx, side="left"); hi = np.searchsorted(T["keys3"], key_ctx + V, side="left")
    C["ctx3"] = np.where(has_u, T["cs3"][hi] - T["cs3"][lo], 0)
    C["n3w_uv"] = np.where(has_u, hi - lo, 0)
    if "keys_a2" in T:
        C["a2"] = look(T["keys_a2"], T["cnt_a2"], v * V + w)
        C["tot_a2_v"] = T["tot_a2"][v]; C["nw_a2_v"] = T["nw_a2"][v]; C["a1_w"] = T["a1"][w]
    return C

def p_unigram_ml(T, C):
    return C["uni_w"] / T["Nu"]

def p_bigram_ml(T, C):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(C["ctx2"] > 0, C["c2"] / np.where(C["ctx2"] > 0, C["ctx2"], 1), np.nan)

def p_trigram_ml(T, C):
    with np.errstate(invalid="ignore", divide="ignore"):
        good = C["has_u"] & (C["ctx3"] > 0)
        return np.where(good, C["c3"] / np.where(good, C["ctx3"], 1), np.nan)

def p_add_k(T, C, order, k):
    if order == 1:
        return (C["uni_w"] + k) / (T["Nu"] + k * V)
    if order == 2:
        return (C["c2"] + k) / (C["ctx2"] + k * V)
    p3 = (C["c3"] + k) / (C["ctx3"] + k * V)
    p2 = (C["c2"] + k) / (C["ctx2"] + k * V)
    return np.where(C["has_u"], p3, p2)

def p_interp(T, C, lam1, lam2, lam3):
    # one weight per order, ending at the uniform 1/|V|
    p1 = lam1 * p_unigram_ml(T, C) + (1 - lam1) / V
    ml2 = np.where(C["ctx2"] > 0, C["c2"] / np.maximum(C["ctx2"], 1), 0.0)
    p2 = np.where(C["ctx2"] > 0, lam2 * ml2 + (1 - lam2) * p1, p1)
    good = C["has_u"] & (C["ctx3"] > 0)
    ml3 = np.where(good, C["c3"] / np.maximum(C["ctx3"], 1), 0.0)
    p3 = np.where(good, lam3 * ml3 + (1 - lam3) * p2, p2)
    return p1, p2, p3

def p_kn(T, C, D=0.75):
    # interpolated Kneser-Ney, one discount D at every order, a uniform 1/|V| at the lowest level (J&M App. C)
    Nu = T["Nu"]
    kn_uni = np.maximum(C["uni_w"] - D, 0) / Nu + D * T["nzu"] / Nu / V
    A1 = T["A1"]
    cont1 = np.maximum(C["a1_w"] - D, 0) / A1 + D * T["nz1"] / A1 / V
    tot = C["tot_a2_v"]
    cont2 = np.where(tot > 0, np.maximum(C["a2"] - D, 0) / np.maximum(tot, 1) + D * C["nw_a2_v"] / np.maximum(tot, 1) * cont1, cont1)
    ctx2 = C["ctx2"]
    kn_bi = np.where(ctx2 > 0, np.maximum(C["c2"] - D, 0) / np.maximum(ctx2, 1) + D * C["n2w_v"] / np.maximum(ctx2, 1) * cont1, cont1)
    ctx3 = C["ctx3"]
    good = C["has_u"] & (ctx3 > 0)
    kn_tri = np.where(good, np.maximum(C["c3"] - D, 0) / np.maximum(ctx3, 1) + D * C["n3w_uv"] / np.maximum(ctx3, 1) * cont2, cont2)
    kn_tri = np.where(C["has_u"], kn_tri, kn_bi)                  # no two-token context at t = 2: the bigram model scores it
    return kn_uni, kn_bi, kn_tri

def stream_components(T, S):
    w, v = S[1:], S[:-1]
    u = np.concatenate([[0], S[:-2]])
    has_u = np.concatenate([[False], np.ones(len(S) - 2, dtype=bool)])
    return components(T, u, v, w, has_u)

def perplexity(p):
    p = np.asarray(p, dtype=np.float64)
    if np.isnan(p).any() or (p <= 0).any():
        return float("inf")
    return float(np.exp(-np.mean(np.log(p))))

def mean_nll(p):
    p = np.asarray(p, dtype=np.float64)
    if np.isnan(p).any() or (p <= 0).any():
        return float("inf")
    return float(-np.mean(np.log(p)))

def four_gram_keys(S):
    k = S[:-3].astype(np.int64); k *= V                       # in place: one int64 array, no temporaries
    k += S[1:-2]; k *= V; k += S[2:-1]; k *= V; k += S[3:]
    return k
print("library defined")

library defined


### Section 3: the table goes sparse
For each training size (the first N tokens): the distinct bigrams and trigrams, and the share of **test** n-grams the table has never seen. The unseen rates use the 272,349 test positions that have three tokens of context, so every order is scored at the same positions.

In [9]:
SIZES_GROWTH = [10**3, 10**4, 10**5, 10**6, 10**7, None]
pos4 = np.arange(3, len(te))
w4, c4, b4, a4 = te[pos4], te[pos4 - 1], te[pos4 - 2], te[pos4 - 3]
q1, q2 = w4, c4 * V + w4
q3 = (b4 * V + c4) * V + w4
q4 = ((a4 * V + b4) * V + c4) * V + w4
growth, unseen = [], []
t0 = time.time()
for m in SIZES_GROWTH:
    m_ = len(tr) if m is None else m
    Tg = build_tables(tr, m_, full=False)
    st = dict(Tg["stats"])
    k4 = four_gram_keys(tr[:m_])
    k4.sort()
    keys4, cnt4 = runs(k4); del k4
    st["distinct_4"] = int(len(keys4))
    growth.append(st)
    us = {"m": int(m_), "N_common": int(len(pos4)),
          "unseen_uni": int((Tg["uni"][q1] == 0).sum()),
          "unseen_bi": int((look(Tg["keys2"], Tg["cnt2"], q2) == 0).sum()),
          "unseen_tri": int((look_cs(Tg["keys3"], Tg["cs3"], q3) == 0).sum()),
          "unseen_4": int((look(keys4, cnt4, q4) == 0).sum())}
    unseen.append(us)
    del keys4, cnt4, Tg
print("growth and unseen rates:", round(time.time() - t0), "s")

growth and unseen rates: 11 s


In [10]:
EXPECTED_GROWTH = [['10,000', '7,090', '0.000281%', '8,917', '82.6%', '96.9%', '99.1%'], ['100,000', '57,148', '0.00226%', '84,829', '62.6%', '90.2%', '97.5%'], ['1,000,000', '382,294', '0.0151%', '738,037', '36.8%', '75.0%', '92.0%'], ['10,000,000', '2,029,516', '0.0804%', '5,731,060', '16.1%', '53.0%', '79.5%'], ['113,296,181', '9,414,048', '0.373%', '41,318,952', '5.4%', '28.9%', '57.2%']]        # the Section 3 table of the episode file, copied at build time
ok(unseen[0]["N_common"] == 272_349)
for exp_row, st, us in zip(EXPECTED_GROWTH, growth[1:], unseen[1:]):
    got = [f"{st['m']:,}", f"{st['distinct_bi']:,}", f"{100 * st['distinct_bi'] / V ** 2:.3g}%", f"{st['distinct_tri']:,}",
           f"{100 * us['unseen_bi'] / us['N_common']:.1f}%", f"{100 * us['unseen_tri'] / us['N_common']:.1f}%", f"{100 * us['unseen_4'] / us['N_common']:.1f}%"]
    ok(got == exp_row, (got, exp_row))
# the 1,000-token row (not printed in the episode) and the full-size counts quoted in the text
g1, u1 = growth[0], unseen[0]
ok((g1["distinct_bi"], g1["distinct_tri"], g1["distinct_4"]) == (849, 946, 978))
ok([round(100 * u1[k] / u1["N_common"], 1) for k in ("unseen_uni", "unseen_bi", "unseen_tri", "unseen_4")] == [56.3, 92.1, 99.0, 99.8])
gf, uf = growth[-1], unseen[-1]
ok(gf["m"] == 113_296_181 and gf["N_pairs"] == 113_296_180 and gf["distinct_bi"] == 9_414_048)    # "our 113,296,180 pairs gave 9,414,048 distinct bigrams"
ok(gf["distinct_4"] == 73_707_102 and gf["tri_singletons"] == 30_552_016)
ok(round(100 * gf["tri_singletons"] / gf["distinct_tri"], 1) == 73.9)                              # "73.9% of the distinct trigrams occurred once"
ok((uf["unseen_uni"], uf["unseen_bi"], uf["unseen_tri"], uf["unseen_4"]) == (8, 14_698, 78_729, 155_838))
ok(growth[-1]["distinct_bi"] < gf["N_pairs"] and gf["distinct_bi"] <= V ** 2)                      # a sparse table holds at most one entry per pair
print("SECTION 3 TABLE MATCHES THE EPISODE:", len(EXPECTED_GROWTH), "rows;", checks, "checks so far")

SECTION 3 TABLE MATCHES THE EPISODE: 5 rows; 439 checks so far


### Section 4: smoothing, and the perplexity table
Perplexity is Episode 4's: exp of the mean negative log-likelihood per scored token (the test stream has N = 272,351 scored targets). **Unsmoothed** counting gives probability 0 (or an undefined 0/0) to unseen pairs, so its perplexity is infinite at every order and size. Add-k's k and the interpolation weights are tuned on the validation stream (never on test); Kneser-Ney's discount is 0.75, fixed in advance.

In [11]:
K_GRID = [1, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001, 5e-4, 2e-4, 1e-4, 5e-5, 2e-5, 1e-5, 1e-6]
LAM_GRID = [0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999, 0.9995, 0.9999]
D_MAIN = 0.75
SIZES_EVAL = [10**5, 10**6, 10**7, None]

def tune_addk(Cv, T, order):
    best = (float("inf"), None)
    for k in K_GRID:
        f = mean_nll(p_add_k(T, Cv, order, k))
        if f < best[0]:
            best = (f, k)
    return best[1], best[0]

def tune_interp(Cv, T, order):
    lam = [0.9, 0.5, 0.5]
    def f(l):
        return mean_nll(p_interp(T, Cv, *l)[order - 1])
    best = f(lam)
    for _ in range(4):
        for i in range(order):
            for g in LAM_GRID:
                trial = list(lam); trial[i] = g
                v = f(trial)
                if v < best - 1e-15:
                    best, lam = v, trial
    return lam[:order], best

perp = []
t0 = time.time()
for m in SIZES_EVAL:
    m_ = len(tr) if m is None else m
    T = build_tables(tr, m_, full=True)
    ok(int(T["uni"].sum()) == m_ - 1 and int(T["c2tot"].sum()) == m_ - 1 and int(T["cs3"][-1]) == m_ - 2)   # N = m - 1 pairs, m - 2 trigrams, every one counted once
    Cv, Ct = stream_components(T, va), stream_components(T, te)
    row = {"m": int(m_)}
    for name, fn in (("ml_uni", p_unigram_ml), ("ml_bi", p_bigram_ml), ("ml_tri", p_trigram_ml)):
        pt = fn(T, Ct)
        if name == "ml_tri":
            pt = pt[Ct["has_u"]]
        bad = np.isnan(pt) | (pt <= 0)
        row[name] = {"of": int(len(pt)), "zero": int(((pt == 0) & ~np.isnan(pt)).sum()), "undefined": int(np.isnan(pt).sum()),
                     "perplexity": "inf" if bad.any() else float(np.exp(-np.mean(np.log(pt))))}
    for order, name in ((1, "addk_uni"), (2, "addk_bi"), (3, "addk_tri")):
        k, fv = tune_addk(Cv, T, order)
        row[name] = {"k": k, "test_perplexity": perplexity(p_add_k(T, Ct, order, k))}
    for order, name in ((1, "interp_uni"), (2, "interp_bi"), (3, "interp_tri")):
        lam, fv = tune_interp(Cv, T, order)
        row[name] = {"lambdas": lam, "test_perplexity": perplexity(p_interp(T, Ct, *(lam + [0.5] * (3 - len(lam))))[order - 1])}
    for D in (D_MAIN, 0.5, 0.9):
        row["kn" if D == D_MAIN else f"kn_D{D}"] = {"D": D, "test_perplexity": [perplexity(x) for x in p_kn(T, Ct, D)]}
    perp.append(row)
    print("size", f"{m_:,}", "done", round(time.time() - t0), "s", flush=True)
    if m is None:
        full_T, full_Cv, full_Ct = T, Cv, Ct
    else:
        del T
T = full_T
print("perplexities computed")

size 100,000 done 3 s


size 1,000,000 done 7 s


size 10,000,000 done 11 s


size 113,296,181 done 21 s


perplexities computed


In [12]:
EXPECTED_PERP = [['Unigram (0.75 discount, uniform base)', '2,697', '2,153', '2,018', '1,993'], ['Bigram, add-k', '6,768', '1,609', '470', '250'], ['Bigram, Kneser-Ney', '1,536', '666', '326', '225'], ['Trigram, add-k', '27,058', '9,936', '2,198', '421'], ['Trigram, interpolation', '1,527', '656', '278', '136'], ['Trigram, Kneser-Ney', '1,520', '600', '241', '118']]          # the Section 4 table of the episode file, copied at build time
def cell(x):
    return f"{x:,.0f}"
perp_rows = {
    "Unigram (0.75 discount, uniform base)": [r["kn"]["test_perplexity"][0] for r in perp],
    "Bigram, add-k": [r["addk_bi"]["test_perplexity"] for r in perp],
    "Bigram, Kneser-Ney": [r["kn"]["test_perplexity"][1] for r in perp],
    "Trigram, add-k": [r["addk_tri"]["test_perplexity"] for r in perp],
    "Trigram, interpolation": [r["interp_tri"]["test_perplexity"] for r in perp],
    "Trigram, Kneser-Ney": [r["kn"]["test_perplexity"][2] for r in perp],
}
for exp_row in EXPECTED_PERP:
    ok([cell(x) for x in perp_rows[exp_row[0]]] == exp_row[1:], (exp_row, [cell(x) for x in perp_rows[exp_row[0]]]))
# the sentences around the table
ok(len(perp_rows) == len(EXPECTED_PERP))
ok(perp[-1]["addk_bi"]["k"] == 0.005 and perp[-1]["addk_tri"]["k"] == 0.001)                  # best k at full size
ok(perp[-1]["interp_bi"]["lambdas"] == [0.9, 0.9] and perp[-1]["interp_tri"]["lambdas"] == [0.8, 0.8, 0.6])
ok(round(0.005 * V) == 251)                                                                     # k = 0.005 adds about 251 to every context's total
for r in perp:                                                                                  # unsmoothed: infinite at every order and size
    ok(r["ml_uni"]["perplexity"] == "inf" or r["ml_uni"]["zero"] > 0)
    ok(r["ml_bi"]["perplexity"] == "inf" and r["ml_tri"]["perplexity"] == "inf")
ok(perp[-1]["ml_bi"]["zero"] == 14_690 and perp[-1]["ml_bi"]["undefined"] == 8 and perp[-1]["ml_bi"]["of"] == 272_351)
ok(perp[-1]["ml_tri"]["zero"] == 64_031 and perp[-1]["ml_tri"]["undefined"] == 14_698 and perp[-1]["ml_tri"]["of"] == 272_350)
ok(abs(math.exp(mean_nll(np.full(1000, 1.0 / V))) - 50_257) < 1e-6)                            # a uniform guess scores |V|
for r in perp:                                                                                  # trigram add-k worse than bigram add-k at every size
    ok(r["addk_tri"]["test_perplexity"] > r["addk_bi"]["test_perplexity"])
edge = [r["kn"]["test_perplexity"][1] / r["kn"]["test_perplexity"][2] for r in perp]
ok(round(edge[0], 2) == 1.01 and round(edge[-1], 2) == 1.90 and edge == sorted(edge))           # Kneser-Ney's trigram edge: 1.01 times to 1.90 times
# discount sensitivity (a research-log sentence): D = 0.9 is lower at the three small sizes, 0.75 is best of the three at full size
d09 = [round(r["kn_D0.9"]["test_perplexity"][2]) for r in perp]
d05 = [round(r["kn_D0.5"]["test_perplexity"][2]) for r in perp]
d75 = [round(r["kn"]["test_perplexity"][2]) for r in perp]
ok(d09 == [1425, 577, 236, 119] and d05 == [1980, 743, 277, 126] and d75 == [1520, 600, 241, 118])
ok(d09[0] < d75[0] and d09[1] < d75[1] and d09[2] < d75[2])
ok(perp[-1]["kn"]["test_perplexity"][2] < perp[-1]["kn_D0.9"]["test_perplexity"][2] < perp[-1]["kn_D0.5"]["test_perplexity"][2])
print("SECTION 4 TABLE MATCHES THE EPISODE:", len(EXPECTED_PERP), "rows;", checks, "checks so far")
print("full size, test perplexity: unigram", round(perp[-1]["kn"]["test_perplexity"][0], 1), " bigram KN", round(perp[-1]["kn"]["test_perplexity"][1], 1),
      " trigram KN", round(perp[-1]["kn"]["test_perplexity"][2], 3))

SECTION 4 TABLE MATCHES THE EPISODE: 6 rows; 472 checks so far
full size, test perplexity: unigram 1992.9  bigram KN 224.7  trigram KN 118.429


### Section 5: the running sentence in the counts
Counts from the whole training stream. After " was" a bigram's answer depends on " was" alone: " cat", " because" and " it" are out of view. A trigram at " it" has no data (" mat because" never occurs), but at " tired" its window holds " it was".

In [13]:
T = full_T
def dec(i): return enc.decode([int(i)])
def top_bigram(v, k=5):
    lo, hi = np.searchsorted(T["keys2"], v * V), np.searchsorted(T["keys2"], (v + 1) * V)
    ks, cn = T["keys2"][lo:hi], T["cnt2"][lo:hi].astype(np.int64)
    ok(bool(np.all(ks // V == v)) and int(cn.sum()) == int(T["c2tot"][v]))        # the slice is exactly the row of v
    o = np.lexsort((ks, -cn))[:k]
    return [(int(ks[i] % V), int(cn[i])) for i in o], int(T["c2tot"][v])
def count_pair(v, w): return int(look(T["keys2"], T["cnt2"], np.array([v * V + w]))[0])
def count_tri(u, v, w): return int(look_cs(T["keys3"], T["cs3"], np.array([(u * V + v) * V + w]))[0])
def ctx_tri(u, v):
    lo, hi = np.searchsorted(T["keys3"], (u * V + v) * V), np.searchsorted(T["keys3"], (u * V + v + 1) * V)
    return int(T["cs3"][hi] - T["cs3"][lo]), int(hi - lo)
THE, MAT, BECAUSE, IT, WAS, TIRED, THEY = 262, 2603, 780, 340, 373, 10032, enc.encode(" they")[0]
ok(dec(THE) == " the" and dec(THEY) == " they" and len(enc.encode(" they")) == 1)
top_the, n_the = top_bigram(THE)
top_bec, n_bec = top_bigram(BECAUSE)
top_was, n_was = top_bigram(WAS)
ok(n_the == 5_574_627 and count_pair(THE, MAT) == 355 and round(100 * 355 / n_the, 4) == 0.0064)    # " mat" 355 of 5,574,627 (0.0064%)
ok(n_was == 1_076_057 and count_pair(WAS, TIRED) == 85 and round(100 * 85 / n_was, 4) == 0.0079)     # " tired": 85 of 1,076,057 (0.0079%)
ok(n_bec == 44_248 and count_pair(BECAUSE, IT) == 4_491 and count_pair(BECAUSE, THEY) == 2_975)
share_bigram = 2_975 / (4_491 + 2_975)
ok(round(share_bigram, 3) == 0.398)                                                                   # the bigram's one share after " because"
ok(ctx_tri(MAT, BECAUSE) == (0, 0))                                                                   # " mat because" never occurs
n_itwas, _ = ctx_tri(IT, WAS)
ok(n_itwas == 60_014 and count_tri(IT, WAS, TIRED) == 2)                                              # a trigram does have data at " tired"
ok(int(T["uni"][TIRED]) == 958)
def fmt_top3(top, tot):
    return ", ".join(f'"{dec(i)}" {100 * c / tot:.1f}%' for i, c in top[:3])
S5 = [['" the" / "The cat sat on the"', '" first" 1.6%, " United" 0.8%, " same" 0.7%', '" floor" 7.6%, " bed" 6.5%, " couch" 5.4%'], ['" because" / "…mat because"', '" of" 28.7%, " the" 12.1%, " it" 10.1%', '" it" 21.7%, " she" 19.1%, " he" 17.9%'], ['" was" / "…because it was"', '" the" 5.1%, " a" 5.0%, " not" 2.5%', '" so" 9.7%, " too" 5.5%, " a" 3.8%'], ['" tired" after " was"', '85 of 1,076,057 (0.0079%)', '2.27% (rank 6)']]
ok(fmt_top3(top_the, n_the) == S5[0][1] and fmt_top3(top_bec, n_bec) == S5[1][1] and fmt_top3(top_was, n_was) == S5[2][1], (S5, fmt_top3(top_was, n_was)))
ok(f"{85:,} of {n_was:,} ({100 * 85 / n_was:.4f}%)" == S5[3][1])
print("bigram column of the Section 5 table matches:", fmt_top3(top_the, n_the), "|", fmt_top3(top_bec, n_bec), "|", fmt_top3(top_was, n_was))

bigram column of the Section 5 table matches: " first" 1.6%, " United" 0.8%, " same" 0.7% | " of" 28.7%, " the" 12.1%, " it" 10.1% | " the" 5.1%, " a" 5.0%, " not" 2.5%


### What the window leaves out, and a cache
For each test target outside the 1,000 most frequent training tokens, did it already occur earlier in **the same paragraph** (one non-empty row), and how far back? Then mix a bag of the previous H tokens of the paragraph into the Kneser-Ney trigram: (1 − w) × p_KN + w × (the share of the previous tokens equal to the target), with w tuned on validation text. A measurement of what a window leaves out, not a published method.

In [14]:
def nearest_lag(S, line_start, dcap=3000):
    n = len(S)
    line_id = np.searchsorted(line_start, np.arange(n), side="right") - 1
    avail = np.arange(n) - line_start[line_id]
    near = np.zeros(n, dtype=np.int64)
    for d in range(1, int(min(dcap, avail.max())) + 1):
        good = (avail[d:] >= d) & (S[d:] == S[:-d]) & (near[d:] == 0)
        near[d:][good] = d
    return near[1:]

def cache_counts(S, line_start, H):
    n = len(S)
    line_id = np.searchsorted(line_start, np.arange(n), side="right") - 1
    avail = np.arange(n) - line_start[line_id]
    hist = np.minimum(H, avail)
    cnt = np.zeros(n, dtype=np.int64)
    for d in range(1, H + 1):
        cnt[d:] += (avail[d:] >= d) & (S[d:] == S[:-d])
    return hist[1:], cnt[1:]

near = nearest_lag(te, ls_te)
wte = te[1:]
cut = np.sort(T["uni"])[::-1][999]
frequent = T["uni"][wte] >= cut
ok(int((T["uni"] >= cut).sum()) == 1000)                                   # exactly 1,000 token types are "frequent"
has_prev = near > 0
rare = ~frequent
n_rare, n_rare_prev = int(rare.sum()), int((rare & has_prev).sum())
n_beyond2, n_beyond10 = int((rare & has_prev & (near > 2)).sum()), int((rare & has_prev & (near > 10)).sum())
ok(len(wte) == 272_351 and int(has_prev.sum()) == 97_824 and round(100 * 97_824 / 272_351, 1) == 35.9)
ok(n_rare == 98_791 and n_rare_prev == 17_583 and round(100 * n_rare_prev / n_rare, 1) == 17.8)    # 17,583 (17.8%) seen earlier in the paragraph
ok(n_beyond2 == 17_533 and round(100 * n_beyond2 / n_rare_prev, 1) == 99.7)                         # 99.7% of those more than two tokens back
ok(n_beyond10 == 15_575 and round(100 * n_beyond10 / n_rare_prev, 1) == 88.6)
ok(float(np.median(near[rare & has_prev])) == 34.0)                                                  # median distance 34
ok(int(np.diff(np.append(ls_te, len(te))).max()) == 526)                                             # the longest paragraph: 526 tokens
MU_GRID = [0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6]
kn3_val = p_kn(T, full_Cv, D_MAIN)[2]
kn3_te = p_kn(T, full_Ct, D_MAIN)[2]
ok(round(perplexity(kn3_te), 1) == 118.4)
cache_res = {}
for H in (10, 50, 200, 1000):
    hv, cv = cache_counts(va, ls_va, H)
    ht, ct = cache_counts(te, ls_te, H)
    pc_v = np.where(hv > 0, cv / np.maximum(hv, 1), 0.0)
    pc_t = np.where(ht > 0, ct / np.maximum(ht, 1), 0.0)
    best = (float("inf"), None)
    for mu in MU_GRID:
        f = mean_nll(np.where(hv > 0, (1 - mu) * kn3_val + mu * pc_v, kn3_val))
        if f < best[0]:
            best = (f, mu)
    mu = best[1]
    cache_res[H] = (mu, perplexity(np.where(ht > 0, (1 - mu) * kn3_te + mu * pc_t, kn3_te)))
ok(cache_res[200][0] == 0.05 and round(cache_res[200][1], 1) == 105.8)                               # 118.4 to 105.8, weight 0.05
ok([round(cache_res[H][1], 2) for H in (10, 50, 200, 1000)] == [116.89, 108.93, 105.78, 105.72])
ok(cache_res[10][0] == 0.005 and cache_res[50][0] == 0.05 and round(100 * (1 - cache_res[200][1] / perplexity(kn3_te)), 1) == 10.7)
print("lag and cache match the episode: 17,583 (17.8%) rare targets seen earlier, 17,533 beyond two tokens; cache 118.4 ->", round(cache_res[200][1], 1))

lag and cache match the episode: 17,583 (17.8%) rare targets seen earlier, 17,533 beyond two tokens; cache 118.4 -> 105.8


### Part 2, last step: Episode 15's 2,048 sentences
The agreement test. Episode 15's sentences end at " because". The bigram's share P(" they")/(P(" it") + P(" they")) there is one number. The trigram also sees the token before " because": the object in template A ("The cat sat on the mat because"), the subject in template B ("On the mat sat the cat because"). AUC: the chance a random plural-subject sentence scores above a random singular one (0.500 is a coin). Scores are Kneser-Ney; add-k is lower.

In [15]:
SUBJ = [('cat','cats'),('dog','dogs'),('bird','birds'),('horse','horses'),('bear','bears'),('goat','goats'),('duck','ducks'),('cow','cows'),
        ('pig','pigs'),('rabbit','rabbits'),('frog','frogs'),('lion','lions'),('monkey','monkeys'),('wolf','wolves'),('mouse','mice'),('snake','snakes')]
OBJ = [('mat','mats'),('box','boxes'),('chair','chairs'),('bed','beds'),('bench','benches'),('step','steps'),('log','logs'),('wall','walls'),
       ('roof','roofs'),('hill','hills'),('rock','rocks'),('stone','stones'),('fence','fences'),('floor','floors'),('table','tables'),('shelf','shelves')]
rows = []
for tpl_ in ('A', 'B'):
    for si_, (s1, s2) in enumerate(SUBJ):
        for sn_ in (0, 1):
            for oi_, (o1, o2) in enumerate(OBJ):
                for on_ in (0, 1):
                    s_, o_ = (s1, s2)[sn_], (o1, o2)[on_]
                    rows.append(dict(tpl=tpl_, sn=sn_, on=on_, text=(f"The {s_} sat on the {o_} because" if tpl_ == 'A' else f"On the {o_} sat the {s_} because")))
ids = [enc.encode(r["text"]) for r in rows]
ok(len(rows) == 2048 and all(len(x) == 7 and x[-1] == 780 for x in ids))
tpl = np.array([r["tpl"] for r in rows]); sn = np.array([r["sn"] for r in rows]); on = np.array([r["on"] for r in rows])
nS = len(rows)
u_arr = np.array([x[-2] for x in ids], dtype=np.int64)
v_arr = np.full(nS, BECAUSE, dtype=np.int64)
def pair_scores(model_fn):
    out = {}
    for name, wid in (("it", IT), ("they", THEY)):
        C = components(T, u_arr, v_arr, np.full(nS, wid, dtype=np.int64), np.ones(nS, dtype=bool))
        out[name] = model_fn(C)
    return out["they"] / (out["they"] + out["it"])
share_bi = pair_scores(lambda C: p_kn(T, C, D_MAIN)[1])
share_tri = pair_scores(lambda C: p_kn(T, C, D_MAIN)[2])
ok(np.all(share_bi == share_bi[0]) and round(float(share_bi[0]), 3) == 0.398)                          # one number for all 2,048 sentences

def auc(score, label):
    x1, x0 = score[label == 1], score[label == 0]
    s0 = np.sort(x0)
    less, leq = np.searchsorted(s0, x1, side="left"), np.searchsorted(s0, x1, side="right")
    return float(((less + leq) / 2).sum() / (len(x1) * len(x0)))
A_, B_ = tpl == 'A', tpl == 'B'
auc_tab = {"Bigram": (auc(share_bi, sn), auc(share_bi, on)),
           "Trigram, template A": (auc(share_tri[A_], sn[A_]), auc(share_tri[A_], on[A_])),
           "Trigram, template B": (auc(share_tri[B_], sn[B_]), auc(share_tri[B_], on[B_]))}
EXPECTED_AUC = [['Bigram', '0.500', '0.500'], ['Trigram, template A', '0.500', '0.605'], ['Trigram, template B', '0.709', '0.500']]
for exp_row in EXPECTED_AUC:
    got = [f"{a:.3f}" for a in auc_tab[exp_row[0]]]
    ok(got == exp_row[1:], (exp_row, got))
ok(auc_tab["Trigram, template B"][0] == 0.708984375 and auc_tab["Trigram, template A"][1] == 0.60546875)    # the exact values
ok(auc_tab["Trigram, template A"][0] == 0.5 and auc_tab["Trigram, template B"][1] == 0.5)                   # by construction (the noun it sees varies independently)
# add-k instead of Kneser-Ney lowers the AUCs; unsmoothed, the share is defined for few sentences
for k_, want_b, want_a in ((0.001, 0.678, 0.568), (0.01, 0.678, 0.557), (0.1, 0.678, 0.565)):
    sh = pair_scores(lambda C: p_add_k(T, C, 3, k_))
    ok(abs(auc(sh[B_], sn[B_]) - want_b) < 0.001 and abs(auc(sh[A_], on[A_]) - want_a) < 0.001, (k_, auc(sh[B_], sn[B_]), auc(sh[A_], on[A_])))
c3_pair = 0                                                                                                 # unsmoothed, the share is defined only where " it" or " they" occurs after this context
for wid in (IT, THEY):
    C = components(T, u_arr, v_arr, np.full(nS, wid, dtype=np.int64), np.ones(nS, dtype=bool))
    c3_pair = c3_pair + C["c3"]
ok(int((c3_pair > 0).sum()) == 416)                                                                            # unsmoothed share defined for only 416 of 2,048
forms = {x[-2]: dec(x[-2]) for x in ids}
ctx_counts = {k: ctx_tri(k, BECAUSE)[0] for k in forms}
ok(len(forms) == 64 and sum(1 for c in ctx_counts.values() if c == 0) == 34)                                # 34 of the 64 noun forms never directly precede " because"
ok(sum(c for c in ctx_counts.values() if c > 0) == 72 and max(ctx_counts.values()) == 7)                    # the other 30 do so 72 times in all (largest 7)
print("AGREEMENT TEST MATCHES THE EPISODE:", {k: tuple(round(a, 3) for a in v) for k, v in auc_tab.items()})

AGREEMENT TEST MATCHES THE EPISODE: {'Bigram': (0.5, 0.5), 'Trigram, template A': (0.5, 0.605), 'Trigram, template B': (0.709, 0.5)}


## Part 3: GPT-2 small
GPT-2 small is run as released (Hugging Face `openai-community/gpt2`, `safetensors` weights; the output layer reuses the token table). Probabilities use a float64 softmax. **One run at temperature 1; not a ranking against the counting models**: the data, the context length and the model class differ, and we state no n-gram figure for GPT-2's own training data.

In [16]:
for pkg in ["safetensors"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import torch
from transformers import GPT2Config, GPT2Model
from safetensors.torch import load_file
torch.set_num_threads(2)
wpath = hf_hub_download("openai-community/gpt2", "model.safetensors", local_dir="gpt2_weights")
model = GPT2Model(GPT2Config())
res = model.load_state_dict(load_file(wpath), strict=False)
model.eval()
ok(not res.missing_keys and all(k.endswith("attn.bias") or k.endswith("masked_bias") for k in res.unexpected_keys))
def logits_for(ids):
    with torch.no_grad():
        h = model(torch.tensor([ids])).last_hidden_state[0]
        return h @ model.wte.weight.T
lg9 = logits_for(sent_ids[:9])
lg5 = logits_for(sent_ids[:5])[-1]
ok(float((lg9[4] - lg5).abs().max()) < 1e-4)                                     # causal: the later tokens do not change an earlier position
def next_dist(pos):
    p = torch.softmax(lg9[pos].double(), -1)
    top = torch.topk(p, 5)
    return p, [(int(i), float(v)) for v, i in zip(top.values, top.indices)]
p_the, top_g_the = next_dist(4)
p_bec, top_g_bec = next_dist(6)
p_was, top_g_was = next_dist(8)
def fmt_g3(top):
    return ", ".join(f'"{dec(i)}" {100 * v:.1f}%' for i, v in top[:3])
ok(fmt_g3(top_g_the) == S5[0][2] and fmt_g3(top_g_bec) == S5[1][2] and fmt_g3(top_g_was) == S5[2][2], (fmt_g3(top_g_the), fmt_g3(top_g_bec), fmt_g3(top_g_was)))
rank_tired = int((p_was > p_was[TIRED]).sum()) + 1
ok(f"{100 * float(p_was[TIRED]):.2f}% (rank {rank_tired})" == S5[3][2])                      # " tired": 2.27% (rank 6)
ok(abs(float(p_the[MAT]) - 0.00404) < 5e-5 and abs(float(p_bec[THEY]) - 0.00877) < 5e-5)     # P(" mat") and P(" they")
ok([dec(i) for i, _ in top_g_the[3:5]] == [" ground", " edge"] and [dec(i) for i, _ in top_g_was[3:5]] == [" hungry", " the"])
print("GPT-2 column of the Section 5 table matches:", fmt_g3(top_g_the), "|", fmt_g3(top_g_bec), "|", fmt_g3(top_g_was), "| tired:", S5[3][2])

GPT-2 column of the Section 5 table matches: " floor" 7.6%, " bed" 6.5%, " couch" 5.4% | " it" 21.7%, " she" 19.1%, " he" 17.9% | " so" 9.7%, " too" 5.5%, " a" 3.8% | tired: 2.27% (rank 6)


In [17]:
# GPT-2's own share P(they)/(P(it) + P(they)) at " because" on Episode 15's 2,048 sentences, and its ranking quality
share_g = np.zeros(nS)
for i in range(0, nS, 256):
    batch = torch.tensor(ids[i:i + 256])
    with torch.no_grad():
        h = model(batch).last_hidden_state[:, -1, :]
        lg = h @ model.wte.weight.T
    share_g[i:i + 256] = torch.sigmoid(lg[:, THEY].double() - lg[:, IT].double()).numpy()
cellmeans = {(t_, s_, o_): float(share_g[(tpl == t_) & (sn == s_) & (on == o_)].mean()) for t_ in "AB" for s_ in (0, 1) for o_ in (0, 1)}   # 8 cells of 256 sentences
ok(all(int(((tpl == t_) & (sn == s_) & (on == o_)).sum()) == 256 for (t_, s_, o_) in cellmeans))
EP14_CELLS = {("A", 0, 0): 0.058, ("A", 0, 1): 0.175, ("A", 1, 0): 0.841, ("A", 1, 1): 0.878,                 # Episode 15's cell means (singular and plural subject, singular and plural object)
              ("B", 0, 0): 0.051, ("B", 0, 1): 0.139, ("B", 1, 0): 0.837, ("B", 1, 1): 0.887}
ok(all(round(cellmeans[k], 3) == v for k, v in EP14_CELLS.items()), {k: round(v, 3) for k, v in cellmeans.items()})
sing = [v for (t_, s_, o_), v in cellmeans.items() if s_ == 0]; plur = [v for (t_, s_, o_), v in cellmeans.items() if s_ == 1]
ok(f"{min(sing):.3f}" == "0.051" and f"{max(sing):.3f}" == "0.175" and f"{min(plur):.3f}" == "0.837" and f"{max(plur):.3f}" == "0.887")   # 0.051 to 0.175; 0.837 to 0.887
ok(np.all(share_g[sn == 1] > 0.5) and np.all(share_g[sn == 0] < 0.5))                          # prefers " they" for every plural-subject sentence, none singular
ok(auc(share_g, sn) == 1.0 and auc(share_g[A_], sn[A_]) == 1.0 and auc(share_g[B_], sn[B_]) == 1.0)       # GPT-2 small's AUC is 1.000
print("GPT-2 small, cell-mean share of ' they' (template, subject number, object number):", {k: round(v, 3) for k, v in cellmeans.items()}, "AUC 1.000")

GPT-2 small, cell-mean share of ' they' (template, subject number, object number): {('A', 0, 0): 0.058, ('A', 0, 1): 0.175, ('A', 1, 0): 0.841, ('A', 1, 1): 0.878, ('B', 0, 0): 0.051, ('B', 0, 1): 0.139, ('B', 1, 0): 0.837, ('B', 1, 1): 0.887} AUC 1.000


In [18]:
# GPT-2 small on the whole test stream, in non-overlapping windows of 1,025 tokens (n_max = 1,024 inputs; the first token of each window is context only)
if RUN_GPT2_TEST:
    WINDOW = 1025
    t0 = time.time()
    nll = np.zeros(len(te)); scored = np.zeros(len(te), dtype=bool)
    starts = list(range(0, len(te), WINDOW))
    ok(len(starts) == 266 and starts[1] == 1025 and len(te) - starts[-1] == 727)                 # 266 windows: 265 of 1,025 tokens and a last one of 727
    for wi, s in enumerate(starts):
        idsw = te[s:s + WINDOW].tolist()
        if len(idsw) < 2:
            continue
        lp = torch.log_softmax(logits_for(idsw[:-1]).double(), -1)
        nll[s + 1:s + len(idsw)] = -lp[torch.arange(len(idsw) - 1), torch.tensor(idsw[1:])].numpy()
        scored[s + 1:s + len(idsw)] = True
        if wi % 50 == 0:
            print("window", wi, "of", len(starts), round(time.time() - t0), "s", flush=True)
    pp_gpt2 = float(np.exp(nll[scored].mean()))
    print("scored", int(scored.sum()), "targets with GPT-2 small in", round(time.time() - t0), "s")
else:
    print("RUN_GPT2_TEST is False: skipped the whole-test-stream comparison (25.738 against 118.426)")

window 0 of 266 2 s


window 50 of 266 93 s


window 100 of 266 185 s


window 150 of 266 278 s


window 200 of 266 369 s


window 250 of 266 460 s


scored 272086 targets with GPT-2 small in 487 s


In [19]:
# the comparison on the same targets (a measurement on one corpus, not a ranking)
if RUN_GPT2_TEST:
    ok(int(scored.sum()) == 272_086)                                                               # 272,086 scored targets
    ok(abs(pp_gpt2 - 25.738) < 0.02, pp_gpt2)                                                      # GPT-2 small: 25.7
    mask = scored[1:]                                                                              # the same targets for the counting models
    kn_all = p_kn(T, full_Ct, D_MAIN)
    same = [perplexity(p[mask]) for p in kn_all]
    ok(abs(same[2] - 118.426) < 5e-3 and abs(same[1] - 224.683) < 5e-3 and abs(same[0] - 1992.844) < 5e-2, same)    # trigram 118.4 on the same targets
    ok(abs(perplexity(kn_all[2]) - 118.429) < 5e-3)                                                # on all 272,351 targets
    print("GPT-2 small", round(pp_gpt2, 3), "against trigram Kneser-Ney", round(same[2], 3), "on the same", int(scored.sum()), "targets (not a ranking)")

GPT-2 small 25.738 against trigram Kneser-Ney 118.426 on the same 272086 targets (not a ranking)


## Back to the Math Lab, on real tokens
Two checks on the real tables. (1) On the first million training tokens, the counts give the lowest training loss of every bigram table we try, and the loss equals the sum over previous tokens of count × H(q) (the identity of Step 1). (2) Our trigram Kneser-Ney, which also mixes in a uniform 1/|V| at the lowest level, sums to 1 over all 50,257 tokens at nine contexts.

In [20]:
m1 = 1_000_000
T1 = build_tables(tr, m1, full=False)
t1 = tr[:m1].astype(np.int64)
prev, nxt = t1[:-1], t1[1:]
ok(len(prev) == m1 - 1)
c_pair = look(T1["keys2"], T1["cnt2"], prev * V + nxt).astype(np.float64)
c_ctx = T1["c2tot"][prev].astype(np.float64)
def L_of(prob): return float(-np.mean(np.log(prob)))
L_ml = L_of(c_pair / c_ctx)
# the group identity: N x L = sum over previous tokens v of count(v) x H(q_v), with q_v the observed next-token mix
cnt2 = T1["cnt2"].astype(np.float64); v_of = T1["keys2"] // V
Hq = -(cnt2 / T1["c2tot"][v_of]) * np.log(cnt2 / T1["c2tot"][v_of])
Hrow = np.bincount(v_of, weights=Hq, minlength=V)
ok(abs((m1 - 1) * L_ml - float((T1["c2tot"] * Hrow).sum())) < 1e-6 * (m1 - 1))
ok(round(L_ml, 4) == 4.0886)
L_k4 = L_of((c_pair + 1e-4) / (c_ctx + 1e-4 * V)); L_k2 = L_of((c_pair + 1e-2) / (c_ctx + 1e-2 * V))
ok(L_k4 > L_ml + 1e-6 and L_k2 > L_k4 + 1e-6)                                                                  # add-k raises the training loss, more for bigger k
# a reweighting of the seen followers inside every row also raises it (a fixed, repeatable reweighting: odd-indexed pairs get 20% more, then renormalise)
w_ = np.where(np.arange(len(T1["keys2"])) % 2 == 1, 1.2, 1.0) * cnt2
rowsum = np.bincount(v_of, weights=w_, minlength=V)
idx_pair = np.searchsorted(T1["keys2"], prev * V + nxt)
ok(L_of(w_[idx_pair] / rowsum[prev]) > L_ml + 1e-6)
print("first 10^6 tokens: L(counts) =", round(L_ml, 6), " add-k 1e-4:", round(L_k4, 4), " add-k 1e-2:", round(L_k2, 4), " reweighted:", round(L_of(w_[idx_pair] / rowsum[prev]), 4))
del T1, t1

first 10^6 tokens: L(counts) = 4.088633  add-k 1e-4: 4.1977  add-k 1e-2: 5.4599  reweighted: 4.0925


In [21]:
# our trigram and bigram Kneser-Ney sum to 1 over all |V| tokens at nine contexts (three from the sentence, six from the test stream)
ctxs = [(319, THE), (MAT, BECAUSE), (IT, WAS)] + [(int(te[i - 1]), int(te[i])) for i in (101, 5_003, 20_017, 77_777, 150_001, 271_999)]
worst = 0.0
for (u_, v_) in ctxs:
    C = components(T, np.full(V, u_, dtype=np.int64), np.full(V, v_, dtype=np.int64), np.arange(V, dtype=np.int64), np.ones(V, dtype=bool))
    for p in p_kn(T, C, D_MAIN):
        worst = max(worst, abs(float(p.sum()) - 1.0))
ok(worst < 1e-9)                                                                                  # "within 10⁻⁹"
worst_other = 0.0                                                                                 # add-k (k = 0.001) and the tuned trigram interpolation sum to 1 as well
for (u_, v_) in ctxs:
    C = components(T, np.full(V, u_, dtype=np.int64), np.full(V, v_, dtype=np.int64), np.arange(V, dtype=np.int64), np.ones(V, dtype=bool))
    worst_other = max(worst_other, abs(float(p_add_k(T, C, 3, 0.001).sum()) - 1.0), abs(float(p_add_k(T, C, 2, 0.005).sum()) - 1.0),
                      abs(float(p_interp(T, C, 0.8, 0.8, 0.6)[2].sum()) - 1.0))
ok(worst_other < 1e-9)
ok(ctx_tri(MAT, BECAUSE)[0] == 0)                                                                  # one of the nine contexts has no trigram data at all
print("Kneser-Ney sums to 1 over all", V, "tokens at", len(ctxs), "contexts; worst deviation", worst)
print()
print("ALL CHECKS PASSED:", checks, "checks")

Kneser-Ney sums to 1 over all 50257 tokens at 9 contexts; worst deviation 2.220446049250313e-16

ALL CHECKS PASSED: 533 checks


## What this notebook did not check
Fetch-based claims in the episode (Shannon, Jurafsky and Martin, Chen and Goodman, Brants et al., Bengio et al.) are quotations and numbers from papers; they are verified in the episode's research log, not here. This notebook reproduces **our** numbers: the counts, the perplexities, the agreement test, the window and cache measurements, the GPT-2 small numbers and the Math Lab. A different corpus, tokenizer, detokenizer or split would give different numbers (without our detokenizer, for example, the test text has 283,287 tokens and the trigram's perplexity is 101.0).